# 同一个成分目标，怎样找全配方？

先不使用零空间这个词。我们从两份用量出发，比较它们的成分，再比较它们的差。

甲、乙、丙每份都是 100 克，两项成分分别是 (10,20)、(20,10)、(30,30) 克。目标是蛋白质 50 克、脂肪 40 克，暂不限制总质量。

依赖 NumPy、Matplotlib。按顺序运行；图使用英文，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
A = np.array([[10., 20., 30.], [20., 10., 30.]])
target = np.array([50., 40.])
x0 = np.array([1., 2., 0.])
x1 = np.array([0., 1., 1.])


## 1. 配方与变化，先分开看

先手算这两份配方的输出。再算“第二份减第一份”。差向量中的负数表示什么？它是否也是一份可用配方？

运行后，注意两份用量的输出是目标，差向量的输出却是零。

In [ ]:
for name, amounts in [("第一份", x0), ("第二份", x1)]:
    print(name, "用量（份）:", amounts, "成分（克）:", A @ amounts,
          "总质量（克）:", 100*amounts.sum())
difference = x1-x0
print("第二份减第一份（份）:", difference)
print("这次调整造成的成分变化（克）:", A @ difference)
assert np.allclose(A @ x0, target)
assert np.allclose(A @ x1, target)
assert np.allclose(A @ difference, [0,0])


## 2. 先选丙，再算甲乙

设丙取 t 份。甲乙剩下的要求是蛋白质 50-30t 克、脂肪 40-30t 克。

先在纸上消元，得到乙为 2-t、甲为 1-t。再预测 t=0、0.5、1 时每一种用量，检查总质量是否也不变。

函数只是执行刚才手推的规则，不是用求解器代替推导。

In [ ]:
def inspect_recipe(t=0.5):
    amounts = np.array([1-t, 2-t, t])
    print("丙取", t, "份")
    print("甲、乙、丙（份）:", amounts)
    print("两项成分（克）:", A @ amounts)
    print("总质量（克）:", 100*amounts.sum())
    print("实际用量都非负:", bool(np.all(amounts >= -1e-10)))
    assert np.allclose(A @ amounts, target)
    return amounts

t = 0.5  # 再试 0、1、1.5、-0.5
chosen = inspect_recipe(t)


## 3. 一个实数解，不一定是一份可用配方

把 t 改成 1.5 和 -0.5。先预测哪种原料用量变成负数。

下图不是成分图：纵轴是用量。为什么三个用量在变，两项成分却不变？为什么只有浅绿色区间能做成实际配方？

In [ ]:
parameters = np.linspace(-0.5, 1.5, 101)
recipes = np.vstack([1-parameters, 2-parameters, parameters])
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
for row, label, color in zip(recipes, ["A amount", "B amount", "C amount"],
                            ["#2878b5", "#d97922", "#359969"]):
    ax.plot(parameters, 100*row, label=label, color=color, lw=2)
ax.axhline(0, color="0.4", lw=0.8)
ax.axvspan(0, 1, color="#359969", alpha=0.12, label="Nonnegative recipes")
ax.set(xlabel="C portions: t", ylabel="Ingredient grams",
       title="Same protein and fat, different ingredient amounts")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
assert np.allclose(A @ recipes, target[:, None])


## 4. 相同的解集，换一份配方作为起点

原来以 (1,2,0) 为起点，加 t(-1,-1,1)。现在改以 (0,1,1) 为起点，加 h(-1,-1,1)。

先想：回到原配方 (1,2,0)，h 应取多少？h 的可用区间还是 [0,1] 吗？

同一配方的两个参数满足 h=t-1。参数改变，不代表解集改变。

In [ ]:
z = np.array([-1., -1., 1.])
h = -0.5  # 再试 -1、0、0.5
from_second_base = x1 + h*z
equivalent_t = h+1
from_first_base = x0 + equivalent_t*z
print("从第二份出发:", from_second_base)
print("对应原来的 t =", equivalent_t, "得到:", from_first_base)
print("成分:", A @ from_second_base)
print("实际用量都非负:", bool(np.all(from_second_base >= -1e-10)))
assert np.allclose(from_second_base, from_first_base)
assert np.allclose(A @ from_second_base, target)
# 以第二份为起点时，非负区间变为 -1 <= h <= 0。


## 5. 随便改变用量，输出都会不变吗？

把调整改为 (0,0,0.5)：只增加丙，不减少甲乙。先预测两项成分会各增加多少。

再改成 (-0.5,-0.5,0.5)。这两次调整的区别是什么？

In [ ]:
adjustment = np.array([0., 0., 0.5])  # 再试 [-0.5,-0.5,0.5]
new_amounts = x0+adjustment
output_change = A @ adjustment
print("调整（份）:", adjustment)
print("成分变化（克）:", output_change)
print("调整后用量（份）:", new_amounts)
print("调整后成分（克）:", A @ new_amounts)
print("是否保持目标:", np.allclose(output_change, 0))
assert np.allclose(A @ new_amounts, target+output_change)


## 6. 现在给零输出调整起名字

所有满足 Az=0 的调整，组成零空间。正文通过消元证明，本例只有 z=t(-1,-1,1) 这一种独立方向。

下面把观察分成两类：

- Ax0=b：已知用量达到目标。
- Az=0：一次调整不改变输出。

由线性关系，A(x0+z)=b；反过来，任意另一个解 x 与 x0 的差都满足 A(x-x0)=0。

实验核对具体数字；“所有解都如此”仍然要用正文的两段推理证明。

In [ ]:
for t in [-2, -0.5, 0, 0.5, 1, 2]:
    adjustment = t*z
    amounts = x0+adjustment
    assert np.allclose(A @ adjustment, [0,0])
    assert np.allclose(A @ amounts, target)
    assert np.allclose(A @ (amounts-x0), [0,0])
print("这些取值核对通过；一般结论由正文推导。")


## 可选滑块

可以直接改 inspect_recipe(t=...) 的参数，不依赖滑块。若已安装 ipywidgets，下方会显示交互控件。

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print("未安装滑块：直接运行 inspect_recipe(t=...)。")
else:
    display(widgets.interactive(inspect_recipe,
        t=widgets.FloatSlider(value=0.5, min=-0.5, max=1.5, step=0.1,
                              continuous_update=False)))


## 合上实验，自己重建

1. (-1,-1,1) 是一份配方吗？它与 (0,1,1) 的作用有什么不同？
2. 不用矩阵符号，从丙的份数推导甲乙用量，说明为什么没有漏解。
3. 用矩阵符号证明：一个已知解加上零输出调整仍是解；任何两个解的差都是零输出调整。
4. 改用 (0,1,1) 作为起点，为什么可用参数变成 -1≤h≤0，配方集合却没变？
5. 如果额外要求总质量固定为 300 克，还能沿原来的方向调整吗？把质量作为第三行加入矩阵，自己检查。
